# Module 3: Selecting Optimal Lag Features Using Genetic Algorithms

## Learning Objectives

By completing this notebook, you will:
1. Understand the role of lag features in time series forecasting
2. Generate comprehensive lag feature sets (univariate and multivariate)
3. Apply GAs to select optimal lag configurations
4. Handle autocorrelation and partial autocorrelation in feature selection
5. Build production-ready autoregressive models with GA-optimized lags

## Prerequisites

- Module 1 completed (GA fundamentals)
- Walk-forward validation notebook completed
- Understanding of autoregression and lag concepts
- DEAP and statsmodels libraries installed

## Estimated Time: 80 minutes

---

## 1. Lag Features in Time Series

### Key Concept: Past values often predict future values in time series.

**What are lag features?**
- Lag 1: Previous time step value
- Lag 2: Value from 2 steps ago
- Lag k: Value from k steps ago

**Why GA for lag selection?**
1. **Curse of dimensionality**: Many possible lags to choose from
2. **Non-linear relationships**: Not all lags are informative
3. **Multivariate complexity**: Multiple features × multiple lags = explosion
4. **Overfitting risk**: Too many lags can harm generalization

**Example:**
```
Original:   [10, 12, 15, 14, 18, 20]
Lag 1:      [NaN, 10, 12, 15, 14, 18]
Lag 2:      [NaN, NaN, 10, 12, 15, 14]
Lag 3:      [NaN, NaN, NaN, 10, 12, 15]
```

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

### 1.1 Generate Time Series with Known Lag Dependencies

In [ ]:
# Purpose: Create time series with specific lag dependencies
# Key Concept: Target depends on lags 1, 2, 7, 14 (daily and weekly patterns)

def generate_lag_dependent_series(n_samples=1000, noise_level=0.1):
    """
    Generate time series where target depends on specific lags.
    
    True dependencies: lag 1, lag 2, lag 7, lag 14
    
    Returns
    -------
    series : pd.Series
        Time series with lag dependencies
    """
    # Initialize series
    series = np.zeros(n_samples)
    
    # Set initial values
    for i in range(14):
        series[i] = np.random.randn()
    
    # Generate series with specific lag dependencies
    for t in range(14, n_samples):
        series[t] = (
            0.5 * series[t-1] +      # Strong lag 1 dependency
            0.3 * series[t-2] +      # Moderate lag 2 dependency
            0.2 * series[t-7] +      # Weekly pattern
            0.15 * series[t-14] +    # Bi-weekly pattern
            np.random.randn() * noise_level
        )
    
    # Convert to pandas series with datetime index
    dates = pd.date_range('2020-01-01', periods=n_samples, freq='D')
    ts = pd.Series(series, index=dates, name='value')
    
    return ts

# Generate series
ts = generate_lag_dependent_series(n_samples=1000, noise_level=0.1)

print(f"Time series length: {len(ts)}")
print(f"Date range: {ts.index[0]} to {ts.index[-1]}")
print(f"\nFirst few values:")
print(ts.head(10))

### 1.2 Visualize Time Series and Autocorrelation

In [ ]:
# Purpose: Visualize time series and identify important lags
# Key Concept: ACF/PACF help identify significant lags

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Time series
axes[0].plot(ts.index, ts.values, linewidth=1.5, color='steelblue')
axes[0].set_title('Time Series', fontsize=14)
axes[0].set_ylabel('Value', fontsize=12)
axes[0].grid(alpha=0.3)

# Plot 2: ACF (Autocorrelation Function)
plot_acf(ts, lags=30, ax=axes[1], alpha=0.05)
axes[1].set_title('Autocorrelation Function (ACF)', fontsize=14)
axes[1].set_xlabel('Lag', fontsize=12)
axes[1].set_ylabel('Correlation', fontsize=12)

# Plot 3: PACF (Partial Autocorrelation Function)
plot_pacf(ts, lags=30, ax=axes[2], alpha=0.05, method='ywm')
axes[2].set_title('Partial Autocorrelation Function (PACF)', fontsize=14)
axes[2].set_xlabel('Lag', fontsize=12)
axes[2].set_ylabel('Partial Correlation', fontsize=12)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - ACF shows total correlation including indirect effects")
print("  - PACF shows direct correlation, controlling for intermediate lags")
print("  - Expect significant spikes at lags 1, 2, 7, 14 (true dependencies)")
print("  - Shaded region represents 95% confidence interval")

## 2. Creating Lag Features

### Key Concept: Transform time series into supervised learning problem with lag features.

In [ ]:
# Purpose: Create comprehensive lag feature set
# Key Concept: Generate multiple lags to give GA many options

def create_lag_features(series, max_lag=30, forecast_horizon=1):
    """
    Create lag features for time series forecasting.
    
    Parameters
    ----------
    series : pd.Series
        Time series data
    max_lag : int
        Maximum lag to create
    forecast_horizon : int
        Steps ahead to forecast (default 1-step ahead)
    
    Returns
    -------
    X : pd.DataFrame
        Lag features
    y : pd.Series
        Target (future values)
    """
    df = pd.DataFrame()
    
    # Create lag features
    for lag in range(1, max_lag + 1):
        df[f'lag_{lag}'] = series.shift(lag)
    
    # Target is future value
    df['target'] = series.shift(-forecast_horizon)
    
    # Drop rows with NaN
    df = df.dropna()
    
    X = df.drop('target', axis=1)
    y = df['target']
    
    return X, y

# Create lag features
MAX_LAG = 30
X_lags, y_target = create_lag_features(ts, max_lag=MAX_LAG, forecast_horizon=1)

print(f"Lag features shape: {X_lags.shape}")
print(f"Target shape: {y_target.shape}")
print(f"\nLag features (first 5 rows):")
print(X_lags.head())
print(f"\nTarget (first 5 values):")
print(y_target.head())

### 2.1 Baseline: Use All Lags

In [ ]:
# Purpose: Establish baseline using all lag features
# Key Concept: Compare against GA-selected lags

from sklearn.model_selection import TimeSeriesSplit

# Time series split
tscv = TimeSeriesSplit(n_splits=5)

rmse_scores_all = []

for train_idx, test_idx in tscv.split(X_lags):
    X_train = X_lags.iloc[train_idx]
    X_test = X_lags.iloc[test_idx]
    y_train = y_target.iloc[train_idx]
    y_test = y_target.iloc[test_idx]
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_scores_all.append(rmse)

avg_rmse_all = np.mean(rmse_scores_all)

print("Baseline Performance (All 30 Lags):")
print("="*70)
print(f"Average RMSE: {avg_rmse_all:.4f}")
print(f"Std RMSE: {np.std(rmse_scores_all):.4f}")
print(f"Min RMSE: {np.min(rmse_scores_all):.4f}")
print(f"Max RMSE: {np.max(rmse_scores_all):.4f}")

## 3. GA-Based Lag Selection

### Key Concept: GA selects subset of lags that optimize forecasting performance.

In [ ]:
# Purpose: Setup DEAP for lag selection
# Key Concept: Each gene represents whether to include a specific lag

# Clean up existing types
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create types
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

print(f"DEAP types created for {MAX_LAG} lag features.")

In [ ]:
# Purpose: Define fitness function for lag selection
# Key Concept: Evaluate using time series cross-validation

def evaluate_lag_selection(individual, X_data, y_data):
    """
    Evaluate subset of lags using time series cross-validation.
    
    Parameters
    ----------
    individual : list
        Binary chromosome (1 = include lag, 0 = exclude)
    X_data : pd.DataFrame
        All lag features
    y_data : pd.Series
        Target values
    
    Returns
    -------
    fitness : tuple
        Negative RMSE (for maximization)
    """
    # Check for at least one lag
    if sum(individual) == 0:
        return (-999.0,)
    
    # Select lags
    lag_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, lag_mask]
    
    # Time series cross-validation
    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []
    
    for train_idx, test_idx in tscv.split(X_selected):
        # Split
        X_train = X_selected.iloc[train_idx]
        X_test = X_selected.iloc[test_idx]
        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train
        model = Ridge(alpha=1.0, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test_scaled)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_scores.append(rmse)
    
    # Average RMSE
    avg_rmse = np.mean(rmse_scores)
    
    # Parsimony penalty: favor fewer lags
    n_lags = sum(individual)
    parsimony_penalty = 0.02 * (n_lags / len(individual))
    
    # Fitness: negative RMSE (maximize = minimize RMSE)
    fitness = -avg_rmse - parsimony_penalty
    
    return (fitness,)

# Test fitness function
# True lags: 1, 2, 7, 14 (indices 0, 1, 6, 13)
test_individual_true = [0] * MAX_LAG
test_individual_true[0] = 1  # lag 1
test_individual_true[1] = 1  # lag 2
test_individual_true[6] = 1  # lag 7
test_individual_true[13] = 1  # lag 14

fitness_true = evaluate_lag_selection(test_individual_true, X_lags, y_target)
print(f"Fitness with true lags (1,2,7,14): {fitness_true[0]:.4f}")

# Test with random lags
test_individual_random = [random.randint(0, 1) for _ in range(MAX_LAG)]
fitness_random = evaluate_lag_selection(test_individual_random, X_lags, y_target)
print(f"Fitness with random lags: {fitness_random[0]:.4f}")

print("\nTrue lags should have better (less negative) fitness!")

### 3.1 Run GA for Lag Selection

In [ ]:
# Purpose: Run GA to find optimal lag configuration
# Key Concept: GA explores lag combinations to minimize forecast error

# Create toolbox
toolbox = base.Toolbox()

# Register components
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=MAX_LAG
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_lag_selection, X_data=X_lags, y_data=y_target)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Statistics
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("max", np.max)
stats.register("min", np.min)
stats.register("std", np.std)

hall_of_fame = tools.HallOfFame(maxsize=5)

# GA parameters
POPULATION_SIZE = 50
MAX_GENERATIONS = 40
P_CROSSOVER = 0.7
P_MUTATION = 0.2

print("Running GA for lag selection...")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Generations: {MAX_GENERATIONS}")
print(f"  Searching among {MAX_LAG} possible lags\n")

# Create population
population = toolbox.population(n=POPULATION_SIZE)

# Run evolution
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=P_CROSSOVER,
    mutpb=P_MUTATION,
    ngen=MAX_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

print("\nGA completed!")

### 3.2 Analyze Selected Lags

In [ ]:
# Purpose: Examine which lags were selected by GA
# Key Concept: GA should identify true lag dependencies (1, 2, 7, 14)

print("Top 3 Solutions:")
print("="*70)

for i, ind in enumerate(hall_of_fame[:3], 1):
    selected_lags = [j+1 for j, gene in enumerate(ind) if gene == 1]
    fitness = ind.fitness.values[0]
    n_lags = sum(ind)
    
    print(f"\nSolution {i}:")
    print(f"  Fitness: {fitness:.4f}")
    print(f"  Number of lags: {n_lags}")
    print(f"  Selected lags: {selected_lags}")

# Best solution
best_individual = hall_of_fame[0]
best_lags = [j+1 for j, gene in enumerate(best_individual) if gene == 1]
best_fitness = best_individual.fitness.values[0]

print(f"\n{'='*70}")
print("Best Solution Analysis:")
print(f"  Selected lags: {best_lags}")
print(f"  True important lags: [1, 2, 7, 14]")
print(f"  Correctly identified: {set(best_lags) & {1, 2, 7, 14}}")
print(f"  Fitness: {best_fitness:.4f}")
print(f"  Baseline (all lags) RMSE: {avg_rmse_all:.4f}")
print(f"  GA-selected RMSE: {-best_fitness:.4f} (approx, includes penalty)")

### 3.3 Visualize Results

In [ ]:
# Purpose: Visualize lag selection and evolution
# Key Concept: Compare selected lags with PACF

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract logbook data
gen = logbook.select("gen")
fit_max = logbook.select("max")
fit_avg = logbook.select("avg")
fit_std = logbook.select("std")

# Plot 1: Fitness evolution
axes[0, 0].plot(gen, fit_max, 'r-', linewidth=2, label='Best')
axes[0, 0].plot(gen, fit_avg, 'b-', linewidth=2, label='Average')
axes[0, 0].fill_between(gen, 
                        np.array(fit_avg) - np.array(fit_std),
                        np.array(fit_avg) + np.array(fit_std),
                        alpha=0.3, color='blue')
axes[0, 0].set_xlabel('Generation', fontsize=11)
axes[0, 0].set_ylabel('Fitness', fontsize=11)
axes[0, 0].set_title('Fitness Evolution', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Selected lags vs PACF
pacf_values = pacf(ts, nlags=MAX_LAG, method='ywm')
lag_indices = np.arange(1, MAX_LAG + 1)

axes[0, 1].bar(lag_indices, np.abs(pacf_values[1:]), color='lightgray', 
              edgecolor='black', alpha=0.5, label='PACF')

# Highlight selected lags
for lag in best_lags:
    axes[0, 1].bar(lag, np.abs(pacf_values[lag]), color='red', 
                  edgecolor='black', alpha=0.8)

axes[0, 1].axhline(y=1.96/np.sqrt(len(ts)), color='blue', 
                   linestyle='--', alpha=0.5, label='95% CI')
axes[0, 1].set_xlabel('Lag', fontsize=11)
axes[0, 1].set_ylabel('|PACF|', fontsize=11)
axes[0, 1].set_title('Selected Lags (Red) vs PACF', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Plot 3: Lag selection frequency across top solutions
lag_counts = np.zeros(MAX_LAG)
for ind in hall_of_fame:
    lag_counts += np.array(ind)
lag_counts /= len(hall_of_fame)

axes[1, 0].bar(lag_indices, lag_counts, color='steelblue', edgecolor='black')
axes[1, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, 
                   label='50% threshold')
axes[1, 0].set_xlabel('Lag', fontsize=11)
axes[1, 0].set_ylabel('Selection Frequency', fontsize=11)
axes[1, 0].set_title('Lag Selection Frequency (Top 5 Solutions)', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Plot 4: Number of lags in final population
n_lags_final = [sum(ind) for ind in population]
axes[1, 1].hist(n_lags_final, bins=15, color='orange', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(sum(best_individual), color='red', linestyle='--', 
                   linewidth=2, label=f'Best: {sum(best_individual)} lags')
axes[1, 1].set_xlabel('Number of Lags', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Lag Count Distribution (Final Population)', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization Interpretation:")
print("  - Top-left: Fitness improves and converges over generations")
print("  - Top-right: Red bars show GA-selected lags, compared to PACF")
print("  - Bottom-left: Consistently selected lags across top solutions")
print("  - Bottom-right: GA converges to sparse lag sets")

## 4. Multivariate Lag Selection

### Key Concept: When multiple time series are available, select lags from all of them.

In [ ]:
# Purpose: Create multivariate time series dataset
# Key Concept: Multiple features, each with potential lag dependencies

def create_multivariate_lag_features(n_samples=1000, n_features=3, max_lag=20):
    """
    Create multivariate time series with lag features.
    
    Returns
    -------
    X : pd.DataFrame
        Lag features from all variables
    y : pd.Series
        Target variable
    """
    # Generate multiple time series
    series_list = []
    
    for i in range(n_features):
        # Each series has different dynamics
        ts_i = generate_lag_dependent_series(n_samples, noise_level=0.1)
        series_list.append(ts_i)
    
    # Create dataframe
    df_multi = pd.DataFrame({
        f'series_{i}': series_list[i] for i in range(n_features)
    })
    
    # Create lag features for each series
    df_lags = pd.DataFrame(index=df_multi.index)
    
    for col in df_multi.columns:
        for lag in range(1, max_lag + 1):
            df_lags[f'{col}_lag_{lag}'] = df_multi[col].shift(lag)
    
    # Target: predict series_0 one step ahead
    df_lags['target'] = df_multi['series_0'].shift(-1)
    
    # Drop NaN
    df_lags = df_lags.dropna()
    
    X = df_lags.drop('target', axis=1)
    y = df_lags['target']
    
    return X, y

# Create multivariate lag features
X_multi, y_multi = create_multivariate_lag_features(n_samples=1000, n_features=3, max_lag=20)

print(f"Multivariate lag features: {X_multi.shape}")
print(f"Total possible lags: {X_multi.shape[1]}")
print(f"\nFeature names (first 10):")
print(X_multi.columns[:10].tolist())

### 4.1 GA for Multivariate Lag Selection

In [ ]:
# Purpose: Run GA on multivariate lag features
# Key Concept: Larger search space, GA becomes even more valuable

# Update toolbox for multivariate case
N_MULTI_LAGS = X_multi.shape[1]

toolbox_multi = base.Toolbox()
toolbox_multi.register("attr_bool", random.randint, 0, 1)
toolbox_multi.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox_multi.attr_bool,
    n=N_MULTI_LAGS
)
toolbox_multi.register("population", tools.initRepeat, list, toolbox_multi.individual)
toolbox_multi.register("evaluate", evaluate_lag_selection, X_data=X_multi, y_data=y_multi)
toolbox_multi.register("mate", tools.cxTwoPoint)
toolbox_multi.register("mutate", tools.mutFlipBit, indpb=0.03)  # Lower mutation rate
toolbox_multi.register("select", tools.selTournament, tournsize=3)

# Statistics
stats_multi = tools.Statistics(key=lambda ind: ind.fitness.values)
stats_multi.register("avg", np.mean)
stats_multi.register("max", np.max)

hall_of_fame_multi = tools.HallOfFame(maxsize=5)

print(f"Running GA for multivariate lag selection...")
print(f"  Search space: {N_MULTI_LAGS} lag features (3 series × 20 lags)")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Generations: 30\n")

# Create and evolve population
population_multi = toolbox_multi.population(n=POPULATION_SIZE)

population_multi, logbook_multi = algorithms.eaSimple(
    population_multi,
    toolbox_multi,
    cxpb=0.7,
    mutpb=0.2,
    ngen=30,
    stats=stats_multi,
    halloffame=hall_of_fame_multi,
    verbose=True
)

print("\nMultivariate GA completed!")

### 4.2 Analyze Multivariate Results

In [ ]:
# Purpose: Examine multivariate lag selection
# Key Concept: See which series and lags are most important

best_ind_multi = hall_of_fame_multi[0]
selected_features_multi = X_multi.columns[np.array(best_ind_multi, dtype=bool)].tolist()

print("Best Multivariate Solution:")
print("="*70)
print(f"Number of lags selected: {sum(best_ind_multi)}/{N_MULTI_LAGS}")
print(f"Fitness: {best_ind_multi.fitness.values[0]:.4f}")
print(f"\nSelected features:")
for feat in selected_features_multi:
    print(f"  - {feat}")

# Count lags by series
series_counts = {f'series_{i}': 0 for i in range(3)}
for feat in selected_features_multi:
    for series_name in series_counts.keys():
        if feat.startswith(series_name):
            series_counts[series_name] += 1
            break

print(f"\nLags selected per series:")
for series_name, count in series_counts.items():
    print(f"  {series_name}: {count} lags")

## 5. Exercises

### Exercise 5.1: Seasonal Lag Selection

**Task**: Modify the lag generation to include seasonal lags (e.g., lag 7, 14, 21 for weekly; lag 30, 60, 90 for monthly). Test if GA discovers seasonal patterns.

**Expected Output**: GA selects seasonal lags when they're important.

<details>
<summary>Hint 1</summary>
Create lags at multiples of the seasonal period: [1, 2, ..., 7, 14, 21, 28, ...].
</details>

<details>
<summary>Hint 2</summary>
Generate time series with strong weekly seasonality: `series[t] = 0.8 * series[t-7] + noise`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_seasonal_lags(series, base_lags=None, seasonal_period=7, n_seasons=4):
    """
    Create lag features including seasonal lags.
    
    Parameters
    ----------
    base_lags : list
        Base lags to include (e.g., [1, 2, 3])
    seasonal_period : int
        Seasonal period (e.g., 7 for weekly)
    n_seasons : int
        Number of seasonal lags (e.g., 4 for 4 weeks)
    """
    # TODO: Implement
    pass

### Exercise 5.2: Hierarchical Lag Selection

**Task**: Implement a two-stage GA: (1) Select which series to include, (2) For selected series, select which lags. Compare with direct selection.

**Expected Output**: Hierarchical approach may converge faster or find better solutions.

<details>
<summary>Hint</summary>
First GA has chromosome length = n_series. Second GA runs on lags of selected series only.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def hierarchical_lag_selection(X_data, y_data, n_series=3, max_lag=20):
    """
    Two-stage GA for lag selection.
    
    Stage 1: Select which series
    Stage 2: Select which lags from selected series
    """
    # TODO: Implement
    pass

### Exercise 5.3: Dynamic Lag Selection

**Task**: Implement walk-forward validation where lag selection is re-run for each window. Analyze how optimal lags change over time.

**Expected Output**: Lag importance may shift as time series dynamics change.

<details>
<summary>Hint</summary>
Run full GA optimization within each walk-forward training window.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def dynamic_lag_selection(X_data, y_data, n_windows=5):
    """
    Re-optimize lag selection for each time window.
    """
    # TODO: Implement walk-forward with GA in each window
    pass

## 6. Summary

### Key Takeaways

1. **Lag Features**: Transform time series into supervised learning with past values
2. **Combinatorial Explosion**: Many lags × many series = huge search space
3. **GA Advantage**: Efficiently explores lag combinations without exhaustive search
4. **ACF/PACF Guidance**: Statistical tests identify candidate lags, GA refines selection
5. **Parsimony**: Fewer lags reduce overfitting and improve generalization
6. **Multivariate**: GA naturally handles cross-series lag dependencies

### Lag Selection Best Practices

- **Start with domain knowledge**: Weekly/monthly patterns suggest specific lags
- **Use ACF/PACF**: Identify statistically significant lags as starting point
- **Parsimony pressure**: Penalize too many lags to prevent overfitting
- **Validation**: Always use time series CV, never standard k-fold
- **Seasonal lags**: Include multiples of seasonal period
- **Computational trade-off**: More lags = slower fitness evaluation

### Comparison: Manual vs GA Lag Selection

| Approach | Pros | Cons |
|----------|------|------|
| **Manual** | Fast, interpretable, domain-informed | Misses interactions, suboptimal |
| **All lags** | Simple, no search needed | Overfitting, slow prediction |
| **Stepwise** | Systematic | Local optima, doesn't consider interactions |
| **GA** | Global search, finds interactions | Slower, needs validation |

### Common Applications

- **Demand forecasting**: Historical sales predict future demand
- **Financial time series**: Past prices, volumes influence future movements
- **Energy consumption**: Usage patterns repeat daily/weekly
- **Climate modeling**: Weather exhibits lag dependencies

### What's Next

- **Next module**: Custom GA operators for feature selection
- **Advanced**: Multi-objective lag selection (accuracy vs complexity)
- **Production**: Deploy lag selection in automated forecasting pipelines

### Additional Resources

- **ARIMA Models**: Box & Jenkins (1976) - Classic time series analysis
- **Feature Engineering**: Kuhn & Johnson (2019) - "Feature Engineering and Selection"
- **GA for Time Series**: Corchado & Fyfe (1999) - "Unsupervised neural method for lag determination"